In [ ]:
### Step 1 — Setup

import sys, os
sys.path.insert(0, os.path.expanduser("~/Sleep_Stage_Research/conference"))

import json
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold
from scipy.stats import wilcoxon

import config as cfg
from data_utils import (compute_class_weights, compute_channel_stats,
                        load_subjects_from_h5, load_participant_info)
from dataset import SleepSequenceDataset
from model import SleepStageNet
from trainer import Trainer
from eval_utils import (predict_on_subjects, compute_fold_metrics, aggregate_cv_results,
                        print_metrics, print_cv_summary, save_results)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

h5_path = os.path.join(cfg.PREPROCESSED_DIR, "dreamt_psg7ch_epochs.h5")
pinfo = load_participant_info()

def assign_age(age):
    if age < 50: return "<50"
    elif age <= 65: return "50-65"
    else: return ">65"

def assign_ahi(ahi):
    if ahi < 5: return "Normal"
    elif ahi < 15: return "Mild"
    elif ahi < 30: return "Moderate"
    else: return "Severe"

pinfo["age_grp"] = pinfo["AGE"].apply(assign_age)
pinfo["ahi_grp"] = pinfo["AHI"].apply(assign_ahi)

COMBOS = []
for age in ["<50", "50-65", ">65"]:
    for ahi in ["Normal", "Mild", "Moderate", "Severe"]:
        subjects = sorted(pinfo[(pinfo["age_grp"] == age) & (pinfo["ahi_grp"] == ahi)]["SID"].tolist())
        label = f"{age.replace('<','lt').replace('>','gt').replace('-','_')}_{ahi}"
        COMBOS.append({"label": label, "subjects": subjects, "age": age, "ahi": ahi})
        print(f"{label}: n={len(subjects)}")

NUM_FOLDS = 3
with open(os.path.join(cfg.CHECKPOINT_DIR, "fold_assignments.json")) as f:
    universal_folds = json.load(f)

exp0_per_subj = {}
for fold_idx in range(cfg.NUM_FOLDS):
    rpath = os.path.join(cfg.CHECKPOINT_DIR, f"exp0_fold{fold_idx}", "test_results.json")
    if os.path.exists(rpath):
        with open(rpath) as f:
            for sid, m in json.load(f)["per_subject"].items():
                exp0_per_subj[sid] = m

FT_LR = 1e-4
FT_MAX_EPOCHS = 20
FT_EARLY_STOP = 7
FT_LR_PATIENCE = 3
EXP_NAME = "exp7_3"
RNN_MODE = "bilstm"

In [ ]:
### Step 2 — FineTuneTrainer

class FineTuneTrainer(Trainer):
    def __init__(self, model, train_loader, val_loader, class_weights,
                 pretrained_path, exp_name, fold, device):
        super().__init__(model, train_loader, val_loader, class_weights,
                         exp_name=exp_name, fold=fold, device=device)
        self.optimizer = torch.optim.Adam(model.parameters(), lr=FT_LR)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode="min", patience=FT_LR_PATIENCE, factor=cfg.LR_FACTOR)
        self.pretrained_path = pretrained_path

    def train(self):
        if self.is_fold_complete():
            history_path = os.path.join(self.save_dir, "train_history.json")
            if os.path.exists(history_path):
                with open(history_path, "r") as f:
                    self.history = json.load(f)
            print(f"  [SKIP] {self.exp_name} fold {self.fold} already complete.")
            return self.history

        resumed = self.load_checkpoint()
        if not resumed:
            ckpt = torch.load(self.pretrained_path, map_location=self.device, weights_only=False)
            remapped = {k.replace("lstm.", "rnn.") if k.startswith("lstm.") else k: v for k, v in ckpt["model_state"].items()}
            self.model.load_state_dict(remapped)
            print(f"  [INIT] Pre-trained model loaded")

        import time
        for epoch in range(self.start_epoch, FT_MAX_EPOCHS):
            t0 = time.time()
            lr = self.optimizer.param_groups[0]["lr"]
            print(f"\n  Epoch {epoch+1}/{FT_MAX_EPOCHS} | lr={lr:.2e}")

            train_loss, train_acc = self.train_one_epoch()
            val_loss, val_acc = self.validate()
            self.scheduler.step(val_loss)

            print(f"  => train_loss={train_loss:.4f} acc={train_acc:.3f} | val_loss={val_loss:.4f} acc={val_acc:.3f} | {time.time()-t0:.0f}s")

            self.history["train_loss"].append(train_loss)
            self.history["val_loss"].append(val_loss)
            self.history["val_acc"].append(val_acc)
            self.history["lr"].append(lr)

            is_best = val_loss < self.best_val_loss
            if is_best:
                self.best_val_loss = val_loss
                self.patience_counter = 0
                print(f"  ** New best val_loss: {val_loss:.4f}")
            else:
                self.patience_counter += 1
                print(f"  Patience: {self.patience_counter}/{FT_EARLY_STOP}")

            self.save_checkpoint(epoch, is_best=is_best)
            if self.patience_counter >= FT_EARLY_STOP:
                print(f"  [EARLY STOP]")
                break

        self.mark_complete()
        with open(os.path.join(self.save_dir, "train_history.json"), "w") as f:
            json.dump(self.history, f, indent=2)
        return self.history

def find_universal_checkpoint(test_subjects):
    min_overlap = len(test_subjects)
    best_fold = 0
    for fold_idx in range(cfg.NUM_FOLDS):
        exp0_test = set(universal_folds[str(fold_idx)]["test"])
        overlap = len(set(test_subjects) & exp0_test)
        if overlap == 0:
            path = os.path.join(cfg.CHECKPOINT_DIR, f"exp0_fold{fold_idx}", "best_model.pt")
            if os.path.exists(path):
                return path, fold_idx
        if overlap < min_overlap:
            min_overlap = overlap
            best_fold = fold_idx
    return os.path.join(cfg.CHECKPOINT_DIR, f"exp0_fold{best_fold}", "best_model.pt"), best_fold

print("Ready.")

In [ ]:
### Step 3 — Train all Age x AHI combinations

all_combo_results = {}

for combo in COMBOS:
    label = combo["label"]
    subjects = combo["subjects"]

    print(f"\n{'='*70}")
    print(f"  {EXP_NAME}: {label} (n={len(subjects)})")
    print(f"{'='*70}")

    if len(subjects) < 3:
        print(f"  [SKIP] Too few subjects ({len(subjects)})")
        continue

    kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=cfg.SEED)
    fold_results = []

    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(subjects)):
        train_subjects = [subjects[i] for i in train_idx]
        test_subjects = [subjects[i] for i in test_idx]

        exp_tag = f"{EXP_NAME}_{label}_fold{fold_idx}"
        print(f"\n{'#'*60}")
        print(f"# {label} | FOLD {fold_idx+1}/{NUM_FOLDS}")
        print(f"{'#'*60}")

        np.random.seed(cfg.SEED + fold_idx)
        n_val = max(1, int(len(train_subjects) * 0.15))
        perm = np.random.permutation(len(train_subjects))
        val_subjects = [train_subjects[i] for i in perm[:n_val]]
        actual_train = [train_subjects[i] for i in perm[n_val:]]

        print(f"  Train: {len(actual_train)} | Val: {len(val_subjects)} | Test: {len(test_subjects)}")

        pretrained_path, univ_fold = find_universal_checkpoint(test_subjects)
        stats = np.load(os.path.join(cfg.CHECKPOINT_DIR, f"exp0_fold{univ_fold}", "channel_stats.npz"))
        mean, std = stats["mean"], stats["std"]

        os.makedirs(os.path.join(cfg.CHECKPOINT_DIR, exp_tag), exist_ok=True)

        _, train_labels, _ = load_subjects_from_h5(h5_path, actual_train)
        class_weights = compute_class_weights(train_labels)
        del train_labels

        train_ds = SleepSequenceDataset(h5_path, actual_train, mean=mean, std=std)
        val_ds = SleepSequenceDataset(h5_path, val_subjects, mean=mean, std=std)

        bs = min(cfg.BATCH_SIZE, len(train_ds) // 2) if len(train_ds) < cfg.BATCH_SIZE * 2 else cfg.BATCH_SIZE
        train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=0, pin_memory=True)

        model = SleepStageNet(n_channels=len(cfg.PSG_CHANNELS), rnn_mode=RNN_MODE).to(device)
        trainer = FineTuneTrainer(model, train_loader, val_loader, class_weights,
                                  pretrained_path=pretrained_path,
                                  exp_name=f"{EXP_NAME}_{label}", fold=fold_idx, device=device)

        trainer.train()

        print(f"  [EVAL] Evaluating on {len(test_subjects)} test subjects...")
        trainer.load_best_model()
        results = predict_on_subjects(model, h5_path, test_subjects, mean, std, device=device)
        overall, per_subj = compute_fold_metrics(results)
        print_metrics(overall, title=f"{label} Fold {fold_idx}")
        fold_results.append((overall, per_subj))

        save_results({"overall": overall, "per_subject": {s: m for s, m in per_subj.items()}},
                     os.path.join(cfg.CHECKPOINT_DIR, exp_tag, "test_results.json"))

        del model, trainer, train_ds, val_ds, train_loader, val_loader
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

    summary = aggregate_cv_results(fold_results)
    all_combo_results[label] = {"summary": summary, "fold_results": fold_results}
    print_cv_summary(summary)
    save_results(summary, os.path.join(cfg.CHECKPOINT_DIR, f"{EXP_NAME}_{label}_cv_summary.json"))

print(f"\n{'='*70}")
print(f"EXP 7.3: ALL AGE x AHI COMPLETE")
print(f"{'='*70}")

In [ ]:
### Step 4 — Summary table

print(f"\n{'='*70}")
print(f"  EXP 7.3: Age x AHI Fine-Tuning Results")
print(f"{'='*70}")

print(f"\n  {'Subgroup':<25} {'n':>4} {'Acc':>8} {'F1':>8} {'κ':>8} {'Univ κ':>8} {'Δκ':>8}")
print(f"  {'-'*69}")

for combo in COMBOS:
    label = combo["label"]
    subjects = combo["subjects"]
    if label not in all_combo_results:
        continue
    s = all_combo_results[label]["summary"]
    u_k = np.mean([exp0_per_subj[sid]["kappa"] for sid in subjects if sid in exp0_per_subj])
    ft_k = s["kappa"]["mean"]
    delta = ft_k - u_k
    sign = "+" if delta > 0 else ""
    print(f"  {label:<25} {len(subjects):>4} {s['accuracy']['mean']:>8.4f} {s['f1_macro']['mean']:>8.4f} "
          f"{ft_k:>8.4f} {u_k:>8.4f} {sign}{delta:>7.4f}")

print(f"\nExperiment 7.3 complete.")